In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Disable auto-scroll in notebook output for smooth interaction
display(HTML("<style>.output_scroll { height: unset !important; }</style>"))

def rational_sampling_time_simulation(L, M):
    clear_output(wait=True)
    
    # 1. Generation of original discrete-time signal x[n]
    n = np.arange(0, 50)  # Αυξήθηκαν τα δείγματα σε 50 για να γεμίζει ο άξονας μέχρι τέρμα
    omega_0 = 0.15 * np.pi
    x = np.cos(omega_0 * n) * np.exp(-0.04 * n)  # damped cosine
    
    # 2. Expander (Interpolation stage): Insert L-1 zeros between samples
    v_len = len(x) * L
    v = np.zeros(v_len)
    v[::L] = x  
    l_indices = np.arange(v_len)
    
    # 3. Intermediate Low-Pass Filter Implementation
    omega_c = min(np.pi / L, np.pi / M)
    filter_half_length = 16
    n_filt = np.arange(-filter_half_length, filter_half_length + 1)
    h = L * (omega_c / np.pi) * np.sinc(omega_c * n_filt / np.pi)
    h = h * np.hamming(len(h))
    
    w = np.convolve(v, h, mode='same')
    
    # 4. Compressor / Decimator stage: Downsample by factor M
    y = w[::M]
    m_indices = np.arange(len(y))
    
    # Plotting results in the time domain
    fig, axes = plt.subplots(3, 1, figsize=(12, 9))
    
    # --- PLOT 1: Original Signal x[n] ---
    axes[0].stem(n, x, linefmt='tab:blue', markerfmt='o', basefmt="k-", label=r'Original Signal $x[n]$ ($F_s$)')
    axes[0].set_title(r'1. Original Input Signal $x[n]$', fontsize=10, fontweight='bold')
    axes[0].set_ylabel('Amplitude', fontsize=9)
    axes[0].set_xlim(-1, 52)  # Επέκταση άξονα μέχρι τέρμα
    axes[0].set_ylim(-1.2, 1.2)
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend(loc='upper right', frameon=True, fontsize=8)

    # --- PLOT 2: Expanded & Filtered Signal w[l] ---
    axes[1].stem(l_indices, w, linefmt='tab:purple', markerfmt='^', basefmt="k-", label=r'Filtered Intermediate Signal $w[\ell]$ ($L \cdot F_s$)')
    axes[1].set_title(fr'2. Expanded ($L={L}$) & Filtered Signal (Cutoff $\min(\pi/L, \pi/M)$)', fontsize=10, fontweight='bold')
    axes[1].set_ylabel('Amplitude', fontsize=9)
    axes[1].set_xlim(-1, 52 * L)  
    axes[1].set_ylim(-1.5 * max(1, L/2), 1.5 * max(1, L/2))
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend(loc='upper right', frameon=True, fontsize=8)

    # --- PLOT 3: Final Decimated Output y[m] ---
    # Map m back to equivalent time base: m * (M/L)
    m_time_axis = m_indices * (M / L)
    axes[2].stem(m_time_axis, y, linefmt='tab:red', markerfmt='s', basefmt="k-", label=r'Final Rate-Converted Output $y[m]$ ($(M/L)F_s$)')
    axes[2].set_title(fr'3. Final Output after Decimation ($M={M}$), Effective Ratio $M/L = {M}/{L} = {M/L:.2f}$', fontsize=10, fontweight='bold')
    axes[2].set_xlabel('Equivalent Time / Sample Index Base', fontsize=9)
    axes[2].set_ylabel('Amplitude', fontsize=9)
    axes[2].set_xlim(-1, 52)  # Επέκταση άξονα μέχρι τέρμα, ίδια με το πρώτο σχήμα
    axes[2].set_ylim(-1.5 * max(1, L/2), 1.5 * max(1, L/2))
    axes[2].grid(True, linestyle='--', alpha=0.6)
    axes[2].legend(loc='upper right', frameon=True, fontsize=8)

    plt.tight_layout()
    plt.show()
    
    print(f"--- Rational Sampling Rate Conversion Summary ---")
    print(f"• Interpolation Factor (L) = {L}")
    print(f"• Decimation Factor (M) = {M}")
    print(f"• Combined Rate Conversion Ratio = M / L = {M} / {L} = {M/L:.4f}")
    print(f"• Dominant Filter Cutoff Frequency = min(pi/{L}, pi/{M}) = {omega_c/np.pi:.3f} * pi")
    if M > L:
        print(f"• Result: Sampling rate is REDUCED (M > L).")
    elif M < L:
        print(f"• Result: Sampling rate is INCREASED (M < L).")
    else:
        print(f"• Result: Sampling rate remains UNCHANGED (M = L).")

# Interactive widgets for L and M sliders
l_slider = widgets.IntSlider(
    value=3, min=1, max=6, step=1,
    description='Interpolation ($L$):',
    style={'description_width': 'initial'}
)

m_slider = widgets.IntSlider(
    value=3, min=1, max=6, step=1,
    description='Decimation ($M$):',
    style={'description_width': 'initial'}
)

ui_lm = widgets.VBox([l_slider, m_slider])
display(ui_lm)

out_lm = widgets.interactive_output(rational_sampling_time_simulation, {'L': l_slider, 'M': m_slider})
display(out_lm)